In [ ]:
#import
%load_ext autoreload
%autoreload 2

import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

import itertools

In [ ]:
#Load Segment Anything

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam_checkpoint = '/home/rthomp12/ObjectPartSeg/sam_vit_h_4b8939.pth'  # Path to the SAM checkpoint
seg_model = load_sam(sam_checkpoint,device)

In [ ]:
#Load the files and camera information
data_file = '/home/rthomp12/fewshot/red_mug_pcl_cleanliness_test/initial_scene.npy'

initial_scene = np.load(data_file, allow_pickle=True).item()
camera_names = initial_scene['cam_names']
camera_names.remove('kevin') # Poor results so we exclude it 
print(f"Camera names: {camera_names}")

rgb_images = initial_scene['rgbs']
depth_images = initial_scene['depths']
intrinsics = initial_scene['intrinsic']
extrinsics = initial_scene['extrinsic']

In [ ]:
#Select the experiment type 

# Mug v Rack
parent_object = 'rack'
child_object = 'mug'

parent_model_files = {'rack': }
child_model_files = {'mug':}


# # Teapot v Mug

# parent_object = 'mug'
# child_object = 'teapot'

# parent_model_files = {'mug': }
# child_model_files = {'teapot':}



# # Bowl v Mug

# parent_object = 'mug'
# child_object = 'bowl'

# parent_model_files = {'mug': }
# child_model_files = {'teapot': }


parent_model = CanonPart.from_pickle(parent_model_file)
child_models = CanonPart.from_pickle(child_model_file)

all_names = [parent_object, child_object]

all_image_points = {camera_name:{name:[] for name in all_names} for camera_name in camera_names}
gripper_points = {camera_name: [] for camera_name in camera_names}

In [ ]:
#Segmentation GUI
part = all_names[0]
camera_name = camera_names[0]

colors = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink']
color = 'red'
points = all_image_points[camera_name][part]
from ipywidgets import widgets 

def on_click(event):
    global color, cup_points, handle_points, points
    ix, iy = int(event.xdata), int(event.ydata)
    print(f"Coordinates: x={ix}, y={iy}")
    points.append([ix, iy])

    # Plot an 'X' marker at the clicked coordinates
    ax.plot(ix, iy, marker='x', markersize=10, color=color, zorder=2)
    fig.canvas.draw()  # Update the figure to show the new marker

def plot_map_with_points():
    global fig, ax, textbox
    
    buttons = widgets.RadioButtons(
        options=all_part_names + ['gripper'],
        disabled=False
    )
    
    display(buttons)

    def radio(value):
        global color, colors, points, all_image_points, camera_name
        part = value['new']
        if part == 'gripper':
            points = gripper_points[camera_name]
            color = colors[len(all_names)]
        else:
            points = all_image_points[camera_name][value['new']]
            color = colors[all_names.index(value['new'])]
        
        
    buttons.observe(radio, names = 'value')
    
    buttons2 = widgets.RadioButtons(
        options=camera_names,
        disabled=False
    )
    
    display(buttons2)
    def radio2(value):
        global points, all_image_points, rgb_images, camera_name, camera_names, part

        camera_name=value['new']
        part_points = all_image_points[camera_name]
        points = part_points[part]
        
        all_image_points[camera_name] = {p:[] for p in all_names}
        gripper_points[camera_name] = []
        ax.imshow(rgb_images[camera_name], cmap=cmap, zorder=1)
        fig.canvas.draw()
        
    buttons2.observe(radio2, names='value')
    
    fig, ax = plt.subplots(figsize=(10, 10))
    cmap = matplotlib.colors.ListedColormap(['black', 'white'])
#     textbox = matplotlib.widgets.TextBox(ax, 'temp',)

    ax.imshow(rgb_images[camera_name], cmap=cmap, zorder=1)

    #plt.legend()
    plt.show()

    # Connect the click event
    cid = fig.canvas.mpl_connect('button_press_event', on_click)

In [ ]:
#Run the segmentation 

%matplotlib widget
plot_map_with_points()

In [ ]:
mask_points, mask_labels, mask_masks, mask_scores, sam_embeddings = {},{},{},{},{}
for dict_starter in [mask_points, mask_labels, mask_masks, mask_scores, sam_embeddings]:
    dict_starter.update({name: {} for name in camera_names})

for camera in camera_names:
    for part in all_names:
    #     print(part)
    #     print(part_points[part])
        mask_points[camera][part] = np.array(all_image_points[camera][part])
        mask_labels[camera][part] = np.ones(mask_points[camera][part].shape[0], dtype=int)
# print(mask_points)

for camera_name in camera_names:
    for obj in all_names:
        seg_points = []
        seg_labels = []
        seg_points += list(mask_points[camera_name][obj])
        seg_labels += list(mask_labels[camera_name][obj])
        for other_part in all_names:
            if obj == other_part:
                continue
            seg_points += list(mask_points[camera_name][other_part])
            seg_labels += list(np.zeros(mask_points[camera_name][other_part].shape[0], dtype=int))

        if len(seg_points) == 0:
            mask_masks[camera_name][obj], mask_scores[camera_name][obj], sam_embeddings[camera_name][obj] = [], [], [],
        else:
            seg_points += list(np.array(gripper_points[camera_name]))
            seg_labels += list(np.zeros(np.array(gripper_points[camera_name]).shape[0], dtype=int))
            mask_masks[camera_name][obj], mask_scores[camera_name][obj], sam_embeddings[camera_name][obj] = segment(seg_model, 
                                                                                                     rgb_images[camera_name], 
                                                                                                     np.array(seg_points), 
                                                                                                     np.array(seg_labels), 
                                                                                                     multimask_flag=False)

In [ ]:
plt.close() #close previous plots

In [ ]:
%matplotlib inline
## Visualize masks
for camera_name in camera_names: 
    for part in all_names:
        for i, (mask, score) in enumerate(zip(mask_masks[camera_name][part], mask_scores[camera_name][part])):
            plt.figure(figsize=(10,10))
            plt.imshow(rgb_images[camera_name])
            show_mask(mask, plt.gca())
            try:
                show_points(mask_points[camera_name][part], mask_labels[camera_name][part], plt.gca())
            except IndexError: #handle is not visible in some views
                continue
            plt.title(f"View, {camera_name}, Mask {part}, Score: {score:.3f}", fontsize=18)
            plt.axis('off')
            plt.show()  

In [ ]:
#PCL post-processing

pcls = {camera_name: get_pointcloud_in_cam_frame(rgb_images[camera_name],
                                                 depth_images[camera_name],
                                                intrinsics[camera_name]).reshape(depth_images[camera_name].shape[0], 
                                                                                 -1, 6) for camera_name in camera_names}
masked_pcls = {part:[] for part in all_names}
for camera_name in ['bob', 'stuart', 'mel', 'dave']: #kevin is cut out bc i'm seeing Bad reflection issues with it:#camera_names:
    for part in all_part_names:
        if len(mask_masks[camera_name][part]) == 0:
            continue
        cloud = pcls[camera_name][mask_masks[camera_name][part][0]]
        if mask_scores[camera_name][part] > .7:
            masked_pcls[part].append(transform_cloud_to_base(cloud, extrinsics, camera_name))
    
for part in all_names:
    try:
        masked_pcls[part] = np.concatenate(masked_pcls[part])[:, :3]
        masked_pcls[part] = masked_pcls[part][masked_pcls[part][:,2] > .01]
        masked_pcls[part] = masked_pcls[part][masked_pcls[part][:,2] < .5]
        masked_pcls[part] = masked_pcls[part][masked_pcls[part][:,1] > -.38]
        masked_pcls[part] = remove_outliers(masked_pcls[part])
    except ValueError:
        continue

In [ ]:
#Visualize PCL

camera = dict(
    eye=dict(x=1.2, y=-1.2, z=.2),
    center=dict(x=.8,y=.5,z=0)
)
viz_utils.show_pcds_plotly({k:v for k,v in masked_pcls.items() if len(v) != 0}, camera=camera
)

In [ ]:
# Visualize pcl by individual camera 
masked_pcls_by_camera = {camera_name:[] for camera_name in camera_names}
for camera_name in camera_names: #kevin is cut out bc i'm seeing Bad reflection issues with it
    for part in all_part_names:
        if len(mask_masks[camera_name][part]) == 0:
            continue
        cloud = pcls[camera_name][mask_masks[camera_name][part][0]]
        if mask_scores[camera_name][part] > .7:
            masked_pcls_by_camera[camera_name].append(transform_cloud_to_base(cloud, extrinsics, camera_name))

for camera_name in camera_names:
    try:
        masked_pcls_by_camera[camera_name] = np.concatenate(masked_pcls_by_camera[camera_name])[:, :3]
        masked_pcls_by_camera[camera_name] = masked_pcls_by_camera[camera_name][masked_pcls_by_camera[camera_name][:,2] > .01]
        masked_pcls_by_camera[camera_name] = masked_pcls_by_camera[camera_name][masked_pcls_by_camera[camera_name][:,2] < .5]
        masked_pcls_by_camera[camera_name] = masked_pcls_by_camera[camera_name][masked_pcls_by_camera[camera_name][:,1] > -.38]
        masked_pcls_by_camera[camera_name] = remove_outliers(masked_pcls_by_camera[camera_name])
    except ValueError:
        continue

camera = dict(
    eye=dict(x=1.2, y=-1.2, z=.2),
    center=dict(x=.8,y=.5,z=0)
)
viz_utils.show_pcds_plotly({k:v for k,v in masked_pcls_by_camera.items() if len(v) != 0}, camera=camera
)

In [ ]:
#Save PCLs
save_name = '/home/rthomp12/fewshot/ridged_mug_flared_rack_grasp/init_scene_pcls'
np.savez(save_name, **masked_pcls)

In [ ]:
#Get PCL Normals (for grasp selection)
masked_pcls = np.load(save_name + '.npz')

part = 'mug'
numpy_pcd = masked_pcls['mug']

# part = 'teapot'
# numpy_pcd = masked_pcls['teapot']

# part = 'bowl'
# numpy_pcd = masked_pcls['bowl']

pcd = o3d.geometry.PointCloud()

pcd.points = o3d.utility.Vector3dVector(numpy_pcd)
pcd.estimate_normals()
pcd = pcd.normalize_normals()

o3d.visualization.draw_geometries([pcd])


In [ ]:
#Import the grasping estimator

from src.GraspSelector import test

In [ ]:
#Visualize the predicted grasps

grasp_index = 0 # higher indices have a higher predicted cost/are less optimal

import plotly.graph_objects as go

data =  [
        go.Scatter3d(
            x=numpy_pcd[:, 0],
            y=numpy_pcd[:, 1],
            z=numpy_pcd[:, 2],
            marker={"size": 5, "color": numpy_pcd[:, 2], "colorscale": "Plotly3"},
            mode="markers",
            opacity=1.0,
        )
    ] + viz_utils.make_axis_plotly(desired_grasp_poses[grasp_index], .1)

fig = go.Figure(data=data)
fig.show()